## Multiple Inputs

This notebook builds on the single-field state from `First_agent.ipynb`. Here the state schema carries **several distinct input fields at once** (`values`, `name`, and later `operation`), and one node reads all of them to compute a `result`. The pattern is the same as before — one `TypedDict`, one `state` argument per node — just with more keys doing real work together.

First Practice

In [ ]:
from typing import TypedDict, List
from langgraph.graph import StateGraph

`List` (from `typing`) lets a state field hold a list with a known element type — here `values: List[int]` — so type checkers and editor hints know `state["values"]` is a list of ints, not just "some list".

In [ ]:
class AgentState(TypedDict):
    values: List[int]
    name: str
    result: str

def process_value(state: AgentState) -> AgentState:
    """This Function handles multiple different inputs"""
    state["result"] = f"Hi there {state['name']}! Your sum = {sum(state["values"])}"
    return state


**Three input fields, one node.** `AgentState` has `values` and `name` as *inputs* and `result` as *output* — `process_value` reads both inputs in the same function and writes the combined answer back into `result`. Nothing new mechanically: it's still `(state) -> state`, just doing more with the extra keys.

Watch the f-string: `f"...{sum(state["values"])}"` reuses double quotes inside a double-quoted f-string. That only works on Python 3.12+ (PEP 701); on older versions it's a `SyntaxError` — use single quotes for the inner key (`state['values']`) if you need broader compatibility.

In [ ]:
graph = StateGraph(AgentState)

graph.add_node("processor", process_value)
graph.set_entry_point("processor")
graph.set_finish_point("processor")

app = graph.compile()

Same builder pattern as `First_agent.ipynb`: bind `StateGraph` to the schema, register the node, mark it as both entry and finish point, then `compile()`. Having more state fields doesn't change the graph wiring at all — it only changes what the node does with them.

In [ ]:
from IPython.display import Image,display
display(Image(app.get_graph().draw_mermaid_png()))

In [ ]:
answers = app.invoke({"values": [1,2,3,4], "name": "Steve"})

**Multiple initial values, one `invoke` call.** All the state's input fields go into the same dict passed to `invoke` — `{"values": [...], "name": "..."}` — exactly like passing multiple keyword arguments, just as one dict instead of separate ones.

In [ ]:
answers["result"]

# second practice

Same idea, one step further: add an `operation` input field and let the node **branch on it** (`+` vs `*`) instead of always doing one fixed computation. The state schema is still just a flat set of fields — branching logic lives entirely inside the node function, not in the graph structure.

In [ ]:
import math
class AgentState2(TypedDict):
    values: List[int]
    name: str
    operation: str
    result: str

def process_value2(state: AgentState2) -> AgentState:
    """This Function handles multiple different inputs depends on if operation mark is + or *"""
    if state["operation"] == "+":
        state["result"] = f"Hi there {state['name']}! Your sum = {sum(state["values"])}"
    elif state["operation"]== "*":
        state["result"] = f"Hi there {state['name']}! Your multiplication = {math.prod(state["values"])}"
    else:
        state["result"]= "choose correct operation between + and *"
    
    return state

`process_value2` checks `state["operation"]` and fills `result` differently for `"+"`, `"*"`, or anything else (a fallback error message) — a simple in-node `if/elif/else` is often all you need before reaching for LangGraph's conditional edges.

Small type-hint slip worth noticing: the function signature says `-> AgentState` but it's built for `AgentState2` (it reads `operation`, which only exists on `AgentState2`). It runs fine either way since Python doesn't enforce return-type annotations at runtime, but the hint should read `-> AgentState2` to match reality.

In [ ]:
graph = StateGraph(AgentState2)

graph.add_node("processor2", process_value2)
graph.set_entry_point("processor2")
graph.set_finish_point("processor2")

app = graph.compile()


In [ ]:
answers = app.invoke({"values": [1,2,3,4], "name": "Steve", "operation": "*"})

Passing `"operation": "*"` here is what steers `process_value2` down the multiplication branch — flip it to `"+"` (or an invalid value) to see the other branches run without touching the graph at all.

In [ ]:
answers["result"]